# Forte Sets: Complete PC-Set Universe

This notebook builds a **complete dataset of all 4096 subsets** of the 12 pitch classes {0, 1, ..., 11}, along with various **relationship links** between them.

## What's Here

### Nodes DataFrame (4096 rows)
Each row represents one subset, identified by its **bitmap integer** (0 to 4095).
- `id_`: Bitmap integer where bit `i` is set iff pitch class `i` is in the set
- `pcset`: Tuple representation, e.g., `(0, 4, 7)` for C major triad
- `cardinality`: Number of pitch classes
- `prime_form`: Canonical representative under T/I equivalence
- `forte_name`: Forte label (e.g., "3-11") if applicable
- `interval_vector`: 6-tuple counting interval classes

### Link DataFrames
- **Immediate subset links**: Hasse diagram of the subset lattice
- **Complement links**: Each set paired with its complement
- **TI-equivalence links**: Sets sharing the same prime form (set class)
- **Z-relation links**: Sets with same interval vector but different prime form
- **R_p similarity links**: Sets sharing n-1 elements under some T/I

### Key Concepts
- **Pitch class (pc)**: Note mod 12 (0=C, 1=C♯, ..., 11=B)
- **Prime form**: Lexicographically smallest form under T_n and I operations
- **Interval vector**: [ic1, ic2, ic3, ic4, ic5, ic6] counts
- **Z-relation**: Same interval content, different prime form
- **Set complex K/Kh**: Reciprocal inclusion relations

In [1]:
import itertools

def contains_zero(pc_set):
    """
    Check if the pitch class set contains 0.
    :param pc_set: set or list of integers representing pitch classes (0-11)
    :return: bool
    """
    return 0 in set(pc_set)

def transpose(pc_set, interval):
    """
    Transpose the pitch class set by a given interval modulo 12.
    :param pc_set: set or list of pitch classes
    :param interval: int, transposition interval
    :return: set of transposed pitch classes
    """
    return {(p + interval) % 12 for p in pc_set}

def invert(pc_set):
    """
    Invert the pitch class set (reflection over 0).
    :param pc_set: set or list of pitch classes
    :return: set of inverted pitch classes
    """
    return {(0 - p) % 12 for p in pc_set}

def normal_form(pc_set):
    """
    Compute the normal form of a pitch class set under transposition.
    :param pc_set: set or list of pitch classes
    :return: tuple, sorted normal form starting with 0
    """
    if not pc_set:
        return ()
    pc_set = set(pc_set)
    candidates = []
    for t in range(12):
        transp = {(p + t) % 12 for p in pc_set}
        sorted_transp = sorted(transp)
        shifted = [(p - sorted_transp[0]) % 12 for p in sorted_transp]
        candidates.append(tuple(sorted(shifted)))
    min_span = min(c[-1] for c in candidates if c)
    packed = [c for c in candidates if c[-1] == min_span]
    return min(packed)

def prime_form(pc_set):
    """
    Compute the prime form of a pitch class set under transposition and inversion.
    :param pc_set: set or list of pitch classes
    :return: tuple, sorted prime form starting with 0
    """
    nf = normal_form(pc_set)
    inv_set = invert(pc_set)
    inv_nf = normal_form(inv_set)
    return min(nf, inv_nf)

def is_transposition_equivalent(set1, set2):
    """
    Check if two sets are equivalent under transposition.
    :param set1: set or list of pitch classes
    :param set2: set or list of pitch classes
    :return: bool
    """
    return normal_form(set1) == normal_form(set2)

def is_ti_equivalent(set1, set2):
    """
    Check if two sets are equivalent under transposition and inversion (set class).
    :param set1: set or list of pitch classes
    :param set2: set or list of pitch classes
    :return: bool
    """
    return prime_form(set1) == prime_form(set2)

def interval_vector(pc_set):
    """
    Compute the interval vector of a pitch class set.
    :param pc_set: set or list of pitch classes
    :return: list of 6 integers, counts of interval classes 1 to 6
    """
    pcs = sorted(set(pc_set))
    n = len(pcs)
    vector = [0] * 6
    for i in range(n):
        for j in range(i + 1, n):
            diff = (pcs[j] - pcs[i]) % 12
            ic = min(diff, 12 - diff)
            vector[ic - 1] += 1
    return vector

def minimal_voice_leading_size(chord1, chord2):
    """
    Compute the minimal voice leading size between two chords (same cardinality), using shortest path mod 12.
    :param chord1: list or set of pitch classes
    :param chord2: list or set of pitch classes
    :return: int, minimal sum of distances
    """
    p1 = list(set(chord1))
    p2 = list(set(chord2))
    if len(p1) != len(p2):
        raise ValueError("Chords must have the same cardinality")
    min_size = float('inf')
    for perm in itertools.permutations(p2):
        size = sum(min(abs(a - b) % 12, 12 - abs(a - b) % 12) for a, b in zip(p1, perm))
        min_size = min(min_size, size)
    return min_size

def is_t_symmetrical(pc_set):
    """
    Check if the set is transpositionally symmetrical (invariant under some non-zero transposition).
    :param pc_set: set or list of pitch classes
    :return: bool
    """
    pc_set = set(pc_set)
    for k in range(1, 12):
        if all((p + k) % 12 in pc_set for p in pc_set):
            return True
    return False

# Dictionary of Forte set classes (prime forms to names)
FORTE_CLASSES = {
    (0,1,2): '3-1',
    (0,1,3): '3-2',
    (0,1,4): '3-3',
    (0,1,5): '3-4',
    (0,1,6): '3-5',
    (0,2,4): '3-6',
    (0,2,5): '3-7',
    (0,2,6): '3-8',
    (0,2,7): '3-9',
    (0,3,6): '3-10',
    (0,3,7): '3-11',
    (0,4,8): '3-12',
    (0,1,2,3): '4-1',
    (0,1,2,4): '4-2',
    (0,1,3,4): '4-3',
    (0,1,2,5): '4-4',
    (0,1,2,6): '4-5',
    (0,1,2,7): '4-6',
    (0,1,4,5): '4-7',
    (0,1,5,6): '4-8',
    (0,1,6,7): '4-9',
    (0,2,3,5): '4-10',
    (0,1,3,5): '4-11',
    (0,2,3,6): '4-12',
    (0,1,3,6): '4-13',
    (0,2,3,7): '4-14',
    (0,1,4,6): '4-15',
    (0,1,5,7): '4-16',
    (0,3,4,7): '4-17',
    (0,1,4,7): '4-18',
    (0,1,4,8): '4-19',
    (0,1,5,8): '4-20',
    (0,2,4,6): '4-21',
    (0,2,4,7): '4-22',
    (0,2,5,7): '4-23',
    (0,2,4,8): '4-24',
    (0,2,6,8): '4-25',
    (0,3,5,8): '4-26',
    (0,2,5,8): '4-27',
    (0,3,6,9): '4-28',
    (0,1,3,7): '4-Z29',
    (0,1,2,3,4): '5-1',
    (0,1,2,3,5): '5-2',
    (0,1,2,4,5): '5-3',
    (0,1,2,3,6): '5-4',
    (0,1,2,3,7): '5-5',
    (0,1,2,5,6): '5-6',
    (0,1,2,6,7): '5-7',
    (0,2,3,4,6): '5-8',
    (0,1,2,4,6): '5-9',
    (0,1,3,4,6): '5-10',
    (0,2,3,4,7): '5-11',
    (0,1,3,5,6): '5-Z12',
    (0,1,2,4,8): '5-13',
    (0,1,2,5,7): '5-14',
    (0,1,2,6,8): '5-15',
    (0,1,3,4,7): '5-16',
    (0,1,3,4,8): '5-Z17',
    (0,1,4,5,7): '5-Z18',
    (0,1,3,6,7): '5-19',
    (0,1,3,7,8): '5-20',
    (0,1,4,5,8): '5-21',
    (0,1,4,7,8): '5-22',
    (0,2,3,5,7): '5-23',
    (0,1,3,5,7): '5-24',
    (0,2,3,5,8): '5-25',
    (0,2,4,5,8): '5-26',
    (0,1,3,5,8): '5-27',
    (0,2,3,6,8): '5-28',
    (0,1,3,6,8): '5-29',
    (0,1,4,6,8): '5-30',
    (0,1,3,6,9): '5-31',
    (0,1,4,6,9): '5-32',
    (0,2,4,6,8): '5-33',
    (0,2,4,6,9): '5-34',
    (0,2,4,7,9): '5-35',
    (0,1,2,4,7): '5-Z36',
    (0,3,4,5,8): '5-Z37',
    (0,1,2,5,8): '5-Z38',
    (0,1,2,3,4,5): '6-1',
    (0,1,2,3,4,6): '6-2',
    (0,1,2,3,5,6): '6-Z3',
    (0,1,2,4,5,6): '6-Z4',
    (0,1,2,3,6,7): '6-5',
    (0,1,2,5,6,7): '6-Z6',
    (0,1,2,6,7,8): '6-7',
    (0,2,3,4,5,7): '6-8',
    (0,1,2,3,5,7): '6-9',
    (0,1,3,4,5,7): '6-Z10',
    (0,1,2,4,5,7): '6-Z11',
    (0,1,2,4,6,7): '6-Z12',
    (0,1,3,4,6,7): '6-Z13',
    (0,1,3,4,5,8): '6-14',
    (0,1,2,4,5,8): '6-15',
    (0,1,4,5,6,8): '6-16',
    (0,1,2,4,7,8): '6-Z17',
    (0,1,2,5,7,8): '6-18',
    (0,1,3,4,7,8): '6-Z19',
    (0,1,4,5,8,9): '6-20',
    (0,2,3,4,6,8): '6-21',
    (0,1,2,4,6,8): '6-22',
    (0,2,3,5,6,8): '6-Z23',
    (0,1,3,4,6,8): '6-Z24',
    (0,1,3,5,6,8): '6-Z25',
    (0,1,3,5,7,8): '6-Z26',
    (0,1,3,4,6,9): '6-Z27',
    (0,1,3,5,6,9): '6-Z28',
    (0,1,3,6,8,9): '6-Z29',
    (0,1,3,6,7,9): '6-30',
    (0,1,3,5,8,9): '6-31',
    (0,2,4,5,7,9): '6-32',
    (0,2,3,5,7,9): '6-33',
    (0,1,3,5,7,9): '6-34',
    (0,2,4,6,8,10): '6-35',
    # Complements can be added if needed, but since prime form is the smaller cardinality one usually
}

def get_forte_name(pc_set):
    """
    Get the Forte name for the pitch class set.
    :param pc_set: set or list of pitch classes
    :return: str, Forte name or None if not found (though all should be)
    """
    pf = prime_form(pc_set)
    return FORTE_CLASSES.get(pf, None)

# Note: For larger sets, use the complement's name with appropriate adjustment if needed.

## Representation Converter

A universal function to translate between different representations of pc-sets:
- **`int`**: Bitmap integer (0 to 4095), where bit `i` is set if pitch class `i` is in the set
- **`tuple`**: Sorted tuple of pitch classes, e.g., `(0, 4, 7)`
- **`frozenset`** / **`set`**: Unordered collection
- **`forte`**: Forte name string, e.g., `"4-19"` (only valid for certain sets)
- **`prime`**: Prime form tuple (canonical representative of equivalence class)

In [3]:
# ---------------------------------------------------------------------------
# REPRESENTATION CONVERTER
# ---------------------------------------------------------------------------

from typing import Union, FrozenSet, Set, Tuple, Optional, Literal

PcSetRepr = Literal['int', 'tuple', 'frozenset', 'set', 'forte', 'prime']

def _int_to_tuple(n: int) -> Tuple[int, ...]:
    """Convert bitmap integer to sorted tuple of pitch classes."""
    return tuple(i for i in range(12) if (n >> i) & 1)

def _tuple_to_int(t) -> int:
    """Convert tuple/set/frozenset of pitch classes to bitmap integer."""
    return sum(1 << p for p in t)

def _int_to_frozenset(n: int) -> FrozenSet[int]:
    """Convert bitmap integer to frozenset."""
    return frozenset(i for i in range(12) if (n >> i) & 1)

# Build reverse lookup: prime_form -> forte_name
PRIME_TO_FORTE = FORTE_CLASSES.copy()

# Also build forte_name -> prime_form for reverse lookup
FORTE_TO_PRIME = {v: k for k, v in FORTE_CLASSES.items()}

def pc_set_convert(
    value,
    to: PcSetRepr,
    *,
    from_repr: Optional[PcSetRepr] = None,
    on_error: Literal['raise', 'none'] = 'raise',
) -> Union[int, Tuple[int, ...], FrozenSet[int], Set[int], str, None]:
    """
    Convert a pitch-class set between representations.
    
    Args:
        value: The pc-set in some representation
        to: Target representation ('int', 'tuple', 'frozenset', 'set', 'forte', 'prime')
        from_repr: Source representation (auto-detected if None)
        on_error: 'raise' to raise ValueError on invalid conversion, 'none' to return None
    
    Returns:
        The pc-set in the target representation, or None if on_error='none' and conversion fails
    
    >>> pc_set_convert(0b10010001, 'tuple')  # bits 0, 4, 7 set -> C major
    (0, 4, 7)
    >>> pc_set_convert((0, 4, 7), 'int')
    145
    >>> pc_set_convert((0, 4, 7), 'forte')
    '3-11'
    >>> pc_set_convert('3-11', 'tuple')
    (0, 3, 7)
    >>> pc_set_convert(0, 'forte', on_error='none')  # empty set has no forte name
    """
    def error(msg):
        if on_error == 'raise':
            raise ValueError(msg)
        return None
    
    # Auto-detect source representation
    if from_repr is None:
        if isinstance(value, int):
            from_repr = 'int'
        elif isinstance(value, str):
            from_repr = 'forte'
        elif isinstance(value, frozenset):
            from_repr = 'frozenset'
        elif isinstance(value, set):
            from_repr = 'set'
        elif isinstance(value, (tuple, list)):
            from_repr = 'tuple'
        else:
            return error(f"Cannot auto-detect representation for {type(value)}")
    
    # First convert to canonical int form
    if from_repr == 'int':
        if not (0 <= value < 4096):
            return error(f"Integer {value} out of range [0, 4095]")
        n = value
    elif from_repr in ('tuple', 'frozenset', 'set'):
        try:
            pcs = set(value)
            if not all(0 <= p < 12 for p in pcs):
                return error(f"Pitch classes must be in [0, 11], got {pcs}")
            n = _tuple_to_int(pcs)
        except (TypeError, ValueError) as e:
            return error(f"Invalid pc-set: {e}")
    elif from_repr == 'forte':
        pf = FORTE_TO_PRIME.get(value)
        if pf is None:
            return error(f"Unknown Forte name: {value}")
        n = _tuple_to_int(pf)
    elif from_repr == 'prime':
        # Prime form is a tuple, convert directly
        n = _tuple_to_int(value)
    else:
        return error(f"Unknown source representation: {from_repr}")
    
    # Now convert from int to target
    if to == 'int':
        return n
    elif to == 'tuple':
        return _int_to_tuple(n)
    elif to == 'frozenset':
        return _int_to_frozenset(n)
    elif to == 'set':
        return set(_int_to_tuple(n))
    elif to == 'forte':
        pcs = _int_to_tuple(n)
        pf = prime_form(pcs)
        # Handle complements: sets of size > 6 use complement's name
        if len(pcs) > 6:
            complement = tuple(i for i in range(12) if i not in pcs)
            pf = prime_form(complement)
        name = PRIME_TO_FORTE.get(pf)
        if name is None:
            return error(f"No Forte name for pc-set {pcs} (prime form {pf})")
        return name
    elif to == 'prime':
        pcs = _int_to_tuple(n)
        if not pcs:
            return ()
        return prime_form(pcs)
    else:
        return error(f"Unknown target representation: {to}")


# Convenience aliases
def int_to_pcset(n: int) -> Tuple[int, ...]:
    """Convert bitmap int to tuple."""
    return _int_to_tuple(n)

def pcset_to_int(pcs) -> int:
    """Convert tuple/set to bitmap int."""
    return _tuple_to_int(pcs)

# Test the converter
print("Converter tests:")
print(f"  145 -> tuple: {pc_set_convert(145, 'tuple')}")
print(f"  (0,4,7) -> int: {pc_set_convert((0,4,7), 'int')}")
print(f"  (0,4,7) -> forte: {pc_set_convert((0,4,7), 'forte')}")
print(f"  '3-11' -> prime: {pc_set_convert('3-11', 'prime')}")

Converter tests:
  145 -> tuple: (0, 4, 7)
  (0,4,7) -> int: 145
  (0,4,7) -> forte: 3-11
  '3-11' -> prime: (0, 3, 7)


## Nodes DataFrame: All 4096 Subsets

Each row represents one of the $2^{12} = 4096$ possible subsets of $\{0, 1, ..., 11\}$.

The **id** column is the bitmap integer representation.

Additional columns capture properties useful for analysis and filtering.

In [4]:
import pandas as pd
import numpy as np

def build_pcset_nodes_df() -> pd.DataFrame:
    """
    Build a DataFrame with all 4096 subsets of {0,...,11}.
    
    Columns:
        id_: Bitmap integer (0 to 4095)
        pcset: Tuple of pitch classes
        cardinality: Number of elements
        contains_zero: Whether 0 is in the set (candidate for normal form)
        complement_id: id of the complement set
        prime_form: Prime form tuple
        forte_name: Forte name (if applicable, else None)
        is_forte_set: Whether this exact set matches a Forte prime form
        interval_vector: 6-tuple of interval class counts
        is_t_symmetric: Transpositionally symmetric
    """
    rows = []
    
    for n in range(4096):
        pcset = _int_to_tuple(n)
        cardinality = len(pcset)
        complement_id = 4095 ^ n  # XOR with all-ones gives complement
        
        # Compute prime form and forte name
        pf = prime_form(pcset) if pcset else ()
        
        # Forte name (only for cardinality 3-6, or complement thereof)
        forte_name = None
        if 3 <= cardinality <= 6:
            forte_name = PRIME_TO_FORTE.get(pf)
        elif 6 < cardinality <= 9:
            # Use complement's name
            comp_pcset = _int_to_tuple(complement_id)
            comp_pf = prime_form(comp_pcset) if comp_pcset else ()
            forte_name = PRIME_TO_FORTE.get(comp_pf)
        
        # Is this set exactly a Forte prime form? (i.e., in normal position)
        is_forte_set = pcset == pf and pf in PRIME_TO_FORTE
        
        # Interval vector
        iv = tuple(interval_vector(pcset)) if pcset else (0, 0, 0, 0, 0, 0)
        
        # Transpositional symmetry
        is_t_sym = is_t_symmetrical(pcset) if cardinality > 0 else False
        
        rows.append({
            'id_': n,
            'pcset': pcset,
            'cardinality': cardinality,
            'contains_zero': 0 in pcset,
            'complement_id': complement_id,
            'prime_form': pf,
            'forte_name': forte_name,
            'is_forte_set': is_forte_set,
            'interval_vector': iv,
            'is_t_symmetric': is_t_sym,
        })
    
    return pd.DataFrame(rows)


# Build the nodes dataframe
nodes_df = build_pcset_nodes_df()
print(f"Nodes DataFrame: {len(nodes_df)} rows")
print(f"\nCardinality distribution:")
print(nodes_df['cardinality'].value_counts().sort_index())
print(f"\nForte sets (prime forms): {nodes_df['is_forte_set'].sum()}")
print(f"Sets with Forte names: {nodes_df['forte_name'].notna().sum()}")
nodes_df.head(10)

Nodes DataFrame: 4096 rows

Cardinality distribution:
cardinality
0       1
1      12
2      66
3     220
4     495
5     792
6     924
7     792
8     495
9     220
10     66
11     12
12      1
Name: count, dtype: int64

Forte sets (prime forms): 114
Sets with Forte names: 3662


,id_,pcset,cardinality,contains_zero,complement_id,prime_form,forte_name,is_forte_set,interval_vector,is_t_symmetric
0,0,(),0,False,4095,(),None,False,"(0, 0, 0, 0, 0, 0)",False
1,1,"(0,)",1,True,4094,"(0,)",None,False,"(0, 0, 0, 0, 0, 0)",False
2,2,"(1,)",1,False,4093,"(0,)",None,False,"(0, 0, 0, 0, 0, 0)",False
3,3,"(0, 1)",2,True,4092,"(0, 1)",None,False,"(1, 0, 0, 0, 0, 0)",False
4,4,"(2,)",1,False,4091,"(0,)",None,False,"(0, 0, 0, 0, 0, 0)",False
5,5,"(0, 2)",2,True,4090,"(0, 2)",None,False,"(0, 1, 0, 0, 0, 0)",False
6,6,"(1, 2)",2,False,4089,"(0, 1)",None,False,"(1, 0, 0, 0, 0, 0)",False
7,7,"(0, 1, 2)",3,True,4088,"(0, 1, 2)",3-1,True,"(2, 1, 0, 0, 0, 0)",False
8,8,"(3,)",1,False,4087,"(0,)",None,False,"(0, 0, 0, 0, 0, 0)",False
9,9,"(0, 3)",2,True,4086,"(0, 3)",None,False,"(0, 0, 1, 0, 0, 0)",False


## Link DataFrames: Relationships Between Sets

Now we define **scalar functions** that measure relationships, and **boolean predicates** that determine if a link exists.

Each link dataframe has columns:
- `source`: id of the first set
- `target`: id of the second set
- (optionally) the scalar value that determined the link

In [ ]:
# ---------------------------------------------------------------------------
# SCALAR FUNCTIONS (measure relationships)
# ---------------------------------------------------------------------------

def shared_pcs_count(set_a, set_b) -> int:
    """Count of shared pitch classes between two sets."""
    return len(set(set_a) & set(set_b))

def iv_difference(iv_a, iv_b) -> int:
    """Sum of absolute differences between two interval vectors."""
    return sum(abs(a - b) for a, b in zip(iv_a, iv_b))

def iv_correspondence(iv_a, iv_b) -> int:
    """Number of matching entries in two interval vectors."""
    return sum(1 for a, b in zip(iv_a, iv_b) if a == b)

def inclusion_bitmask(set_a, set_b) -> int:
    """
    Compute 4-bit inclusion mask for K/Kh complex membership.
    
    Bit 0: A ⊂ B
    Bit 1: A ⊂ B' (complement of B)
    Bit 2: B ⊂ A
    Bit 3: B ⊂ A' (complement of A)
    
    Returns integer 0-15.
    """
    a = set(set_a)
    b = set(set_b)
    universe = set(range(12))
    a_comp = universe - a
    b_comp = universe - b
    
    mask = 0
    if a and b and a < b:  # proper subset
        mask |= 0b0001
    if a and b_comp and a < b_comp:
        mask |= 0b0010
    if b and a and b < a:
        mask |= 0b0100
    if b and a_comp and b < a_comp:
        mask |= 0b1000
    
    return mask

def max_common_subset_size(set_a, set_b) -> int:
    """
    Maximum size of subset shared between two sets under any T_n or I T_n.
    (For R_p relation)
    """
    if len(set_a) != len(set_b):
        return 0
    
    a = frozenset(set_a)
    max_common = 0
    
    for n in range(12):
        # T_n
        b_tn = frozenset((p + n) % 12 for p in set_b)
        common = len(a & b_tn)
        max_common = max(max_common, common)
        
        # I T_n  
        b_itn = frozenset((n - p) % 12 for p in set_b)
        common = len(a & b_itn)
        max_common = max(max_common, common)
    
    return max_common


# ---------------------------------------------------------------------------
# BOOLEAN PREDICATES (derive links from scalars)
# ---------------------------------------------------------------------------

def is_strict_subset(id_a: int, id_b: int) -> bool:
    """True if set A is a strict subset of set B (as bitmaps)."""
    return (id_a & id_b) == id_a and id_a != id_b

def is_immediate_subset(id_a: int, id_b: int) -> bool:
    """True if A ⊂ B and |B| = |A| + 1 (covering relation in subset lattice)."""
    if not is_strict_subset(id_a, id_b):
        return False
    diff = id_b ^ id_a
    return diff != 0 and (diff & (diff - 1)) == 0  # exactly one bit difference

def are_z_related(pf_a, pf_b, iv_a, iv_b) -> bool:
    """True if sets have same interval vector but different prime forms."""
    return iv_a == iv_b and pf_a != pf_b

def is_ti_equivalent_link(pf_a, pf_b) -> bool:
    """True if sets are TI-equivalent (same prime form)."""
    return pf_a == pf_b and pf_a != ()

def is_complement_pair(id_a: int, id_b: int) -> bool:
    """True if A and B are complements."""
    return id_a ^ id_b == 4095

def is_k_complex_member(mask: int) -> bool:
    """True if inclusion mask indicates K complex membership (at least one inclusion)."""
    return mask > 0

def is_kh_subcomplex_member(mask: int) -> bool:
    """True if inclusion mask is 0b1111 (all four inclusions hold)."""
    return mask == 0b1111

def is_rp_similar(common_size: int, set_size: int) -> bool:
    """True if R_p similar (share n-1 elements under some transformation)."""
    return set_size > 1 and common_size == set_size - 1

def is_r1_r2_similar(iv_correspondence: int) -> bool:
    """True if maximally similar (4 of 6 IV entries match)."""
    return iv_correspondence == 4

def is_r0_similar(iv_correspondence: int) -> bool:
    """True if minimally similar (no IV entries match)."""
    return iv_correspondence == 0


print("Scalar and boolean functions defined.")

Scalar and boolean functions defined.


In [6]:
# ---------------------------------------------------------------------------
# LINK DATAFRAME GENERATORS
# ---------------------------------------------------------------------------

def build_subset_links(nodes_df: pd.DataFrame, *, immediate_only: bool = False) -> pd.DataFrame:
    """
    Build links for strict subset relation.
    
    Args:
        nodes_df: The nodes dataframe
        immediate_only: If True, only include covering relations (Hasse diagram edges)
    
    Returns:
        DataFrame with columns: source, target
    """
    links = []
    check = is_immediate_subset if immediate_only else is_strict_subset
    
    for id_a in range(4096):
        for id_b in range(4096):
            if check(id_a, id_b):
                links.append({'source': id_a, 'target': id_b})
    
    return pd.DataFrame(links)


def build_complement_links(nodes_df: pd.DataFrame) -> pd.DataFrame:
    """Build links between sets and their complements."""
    links = [{'source': n, 'target': 4095 ^ n} for n in range(2048)]  # only half to avoid duplicates
    return pd.DataFrame(links)


def build_z_relation_links(nodes_df: pd.DataFrame) -> pd.DataFrame:
    """Build links for Z-related sets (same IV, different prime form)."""
    links = []
    
    # Group by interval vector
    iv_groups = nodes_df.groupby('interval_vector').groups
    
    for iv, indices in iv_groups.items():
        if len(indices) < 2:
            continue
        
        group_rows = nodes_df.loc[indices]
        prime_forms = group_rows['prime_form'].unique()
        
        # Only Z-related if multiple distinct prime forms share same IV
        if len(prime_forms) > 1:
            ids = group_rows['id_'].tolist()
            for i, id_a in enumerate(ids):
                for id_b in ids[i+1:]:
                    pf_a = nodes_df.loc[nodes_df['id_'] == id_a, 'prime_form'].iloc[0]
                    pf_b = nodes_df.loc[nodes_df['id_'] == id_b, 'prime_form'].iloc[0]
                    if pf_a != pf_b:
                        links.append({'source': id_a, 'target': id_b, 'interval_vector': iv})
    
    return pd.DataFrame(links) if links else pd.DataFrame(columns=['source', 'target', 'interval_vector'])


def build_ti_equivalence_links(nodes_df: pd.DataFrame) -> pd.DataFrame:
    """Build links for TI-equivalent sets (same prime form)."""
    links = []
    
    # Group by prime form
    pf_groups = nodes_df[nodes_df['cardinality'] > 0].groupby('prime_form').groups
    
    for pf, indices in pf_groups.items():
        if len(indices) < 2:
            continue
        ids = nodes_df.loc[indices, 'id_'].tolist()
        for i, id_a in enumerate(ids):
            for id_b in ids[i+1:]:
                links.append({'source': id_a, 'target': id_b})
    
    return pd.DataFrame(links) if links else pd.DataFrame(columns=['source', 'target'])


print("Link generators defined.")

Link generators defined.


### Generate Link DataFrames

Let's generate several link dataframes. Note: Some computations are expensive for all 4096×4096 pairs, so we'll use optimizations.

In [7]:
# Fast subset links using bit operations
def build_immediate_subset_links_fast() -> pd.DataFrame:
    """
    Build Hasse diagram edges (immediate subset / covering relation).
    Uses efficient bit manipulation.
    """
    links = []
    for id_a in range(4096):
        # For each set, find supersets that differ by exactly one element
        for bit in range(12):
            if not (id_a & (1 << bit)):  # bit not set in A
                id_b = id_a | (1 << bit)  # add this bit
                links.append({'source': id_a, 'target': id_b})
    
    return pd.DataFrame(links)


# Generate the main link dataframes
print("Building link dataframes...")

# 1. Immediate subset links (Hasse diagram of subset lattice)
print("  Building immediate subset links...")
immediate_subset_links = build_immediate_subset_links_fast()
print(f"    {len(immediate_subset_links)} edges")

# 2. Complement links
print("  Building complement links...")
complement_links = build_complement_links(nodes_df)
print(f"    {len(complement_links)} edges")

# 3. TI-equivalence links (same set class)
print("  Building TI-equivalence links...")
ti_equiv_links = build_ti_equivalence_links(nodes_df)
print(f"    {len(ti_equiv_links)} edges")

# 4. Z-relation links
print("  Building Z-relation links...")
z_relation_links = build_z_relation_links(nodes_df)
print(f"    {len(z_relation_links)} edges")

print("\nDone!")

Building link dataframes...
  Building immediate subset links...
    24576 edges
  Building complement links...
    2048 edges
  Building TI-equivalence links...
    40594 edges
  Building Z-relation links...
    8796 edges

Done!


## Visualization with Force-Directed Graph

Let's visualize the subset lattice using a force-directed graph.

You'll need `cosmograph` for this part. To get it: `pip install cosmograph`

In [21]:
from cosmograph import cosmo

In [ ]:
# Visualize with cosmograph - the subset lattice Hasse diagram
print("Visualizing subset lattice with cosmograph...")
g1 = cosmo(
    points=nodes_df,
    links=immediate_subset_links,
    point_id_by='id_',
    link_source_by='source',
    link_target_by='target',
    point_size_by='cardinality',
    point_color_by='cardinality',
)


Visualizing subset lattice with cosmograph...


Cosmograph(background_color=None, components_display_state_mode=None, focused_point_ring_color=None, hovered_p…

## Additional Link Types

Let's add more sophisticated relationship links.

In [12]:
# ---------------------------------------------------------------------------
# K and Kh COMPLEX LINKS
# ---------------------------------------------------------------------------

def build_k_kh_links(nodes_df: pd.DataFrame, *, kh_only: bool = False) -> pd.DataFrame:
    """
    Build K or Kh complex links.
    
    For K: at least one inclusion relation holds among (A⊂B, A⊂B', B⊂A, B⊂A')
    For Kh: all four must hold (very restrictive)
    
    Args:
        nodes_df: The nodes dataframe
        kh_only: If True, only return Kh links (all 4 conditions met)
    """
    links = []
    n = len(nodes_df)
    
    # Pre-extract pcsets for speed
    pcsets = [set(row['pcset']) for _, row in nodes_df.iterrows()]
    ids = nodes_df['id_'].tolist()
    
    threshold = 0b1111 if kh_only else 1
    check = (lambda m: m == 0b1111) if kh_only else (lambda m: m > 0)
    
    # Only check pairs where cardinalities could allow inclusion
    for i in range(n):
        for j in range(i + 1, n):
            mask = inclusion_bitmask(pcsets[i], pcsets[j])
            if check(mask):
                links.append({
                    'source': ids[i],
                    'target': ids[j],
                    'inclusion_mask': mask,
                })
    
    return pd.DataFrame(links) if links else pd.DataFrame(columns=['source', 'target', 'inclusion_mask'])


# This is expensive for full 4096x4096, so we'll subset
print("Building K-complex links for sets with cardinality 3-6...")

# Filter to interesting cardinalities
forte_range_df = nodes_df[(nodes_df['cardinality'] >= 3) & (nodes_df['cardinality'] <= 9)]
print(f"  Working with {len(forte_range_df)} sets")

# For demo, just show Kh (most restrictive)
# kh_links = build_k_kh_links(forte_range_df, kh_only=True)
# print(f"  Kh links: {len(kh_links)}")


# ---------------------------------------------------------------------------
# R_p SIMILARITY LINKS (same cardinality, share n-1 elements under some T/I)
# ---------------------------------------------------------------------------

def build_rp_similarity_links(nodes_df: pd.DataFrame, cardinality: int = 3) -> pd.DataFrame:
    """
    Build R_p similarity links for sets of a given cardinality.
    
    Two sets are R_p similar if they share n-1 pitch classes under some T_n or I T_n.
    """
    subset = nodes_df[nodes_df['cardinality'] == cardinality]
    pcsets = [tuple(row['pcset']) for _, row in subset.iterrows()]
    ids = subset['id_'].tolist()
    
    links = []
    for i, (id_a, pcs_a) in enumerate(zip(ids, pcsets)):
        for j, (id_b, pcs_b) in enumerate(zip(ids[i+1:], pcsets[i+1:]), start=i+1):
            common = max_common_subset_size(pcs_a, pcs_b)
            if common == cardinality - 1:
                links.append({
                    'source': id_a,
                    'target': id_b,
                    'max_common': common,
                })
    
    return pd.DataFrame(links) if links else pd.DataFrame(columns=['source', 'target', 'max_common'])


print("\nBuilding R_p similarity links for triads (cardinality 3)...")
rp_triads = build_rp_similarity_links(nodes_df, cardinality=3)
print(f"  R_p triad links: {len(rp_triads)}")

print("\nBuilding R_p similarity links for tetrads (cardinality 4)...")
rp_tetrads = build_rp_similarity_links(nodes_df, cardinality=4)
print(f"  R_p tetrad links: {len(rp_tetrads)}")

Building K-complex links for sets with cardinality 3-6...
  Working with 3938 sets

Building R_p similarity links for triads (cardinality 3)...
  R_p triad links: 19872

Building R_p similarity links for tetrads (cardinality 4)...
  R_p tetrad links: 85968


## Summary of Available Data

Let's review what we've built:

In [13]:
print("=" * 60)
print("NODES DATAFRAME")
print("=" * 60)
print(f"Total rows: {len(nodes_df)}")
print(f"\nColumns: {list(nodes_df.columns)}")
print(f"\nSample rows:")
display(nodes_df[nodes_df['cardinality'].isin([3, 4])].head(10))

print("\n" + "=" * 60)
print("LINK DATAFRAMES")
print("=" * 60)

link_summary = {
    'immediate_subset_links': immediate_subset_links,
    'complement_links': complement_links,
    'ti_equiv_links': ti_equiv_links,
    'z_relation_links': z_relation_links,
    'rp_triads': rp_triads,
    'rp_tetrads': rp_tetrads,
}

for name, df in link_summary.items():
    print(f"\n{name}: {len(df)} edges")
    if len(df) > 0:
        print(f"  Columns: {list(df.columns)}")

NODES DATAFRAME
Total rows: 4096

Columns: ['id_', 'pcset', 'cardinality', 'contains_zero', 'complement_id', 'prime_form', 'forte_name', 'is_forte_set', 'interval_vector', 'is_t_symmetric']

Sample rows:


,id_,pcset,cardinality,contains_zero,complement_id,prime_form,forte_name,is_forte_set,interval_vector,is_t_symmetric
7,7,"(0, 1, 2)",3,True,4088,"(0, 1, 2)",3-1,True,"(2, 1, 0, 0, 0, 0)",False
11,11,"(0, 1, 3)",3,True,4084,"(0, 1, 3)",3-2,True,"(1, 1, 1, 0, 0, 0)",False
13,13,"(0, 2, 3)",3,True,4082,"(0, 1, 3)",3-2,False,"(1, 1, 1, 0, 0, 0)",False
14,14,"(1, 2, 3)",3,False,4081,"(0, 1, 2)",3-1,False,"(2, 1, 0, 0, 0, 0)",False
15,15,"(0, 1, 2, 3)",4,True,4080,"(0, 1, 2, 3)",4-1,True,"(3, 2, 1, 0, 0, 0)",False
19,19,"(0, 1, 4)",3,True,4076,"(0, 1, 4)",3-3,True,"(1, 0, 1, 1, 0, 0)",False
21,21,"(0, 2, 4)",3,True,4074,"(0, 2, 4)",3-6,True,"(0, 2, 0, 1, 0, 0)",False
22,22,"(1, 2, 4)",3,False,4073,"(0, 1, 3)",3-2,False,"(1, 1, 1, 0, 0, 0)",False
23,23,"(0, 1, 2, 4)",4,True,4072,"(0, 1, 2, 4)",4-2,True,"(2, 2, 1, 1, 0, 0)",False
25,25,"(0, 3, 4)",3,True,4070,"(0, 1, 4)",3-3,False,"(1, 0, 1, 1, 0, 0)",False



LINK DATAFRAMES

immediate_subset_links: 24576 edges
  Columns: ['source', 'target']

complement_links: 2048 edges
  Columns: ['source', 'target']

ti_equiv_links: 40594 edges
  Columns: ['source', 'target']

z_relation_links: 8796 edges
  Columns: ['source', 'target', 'interval_vector']

rp_triads: 19872 edges
  Columns: ['source', 'target', 'max_common']

rp_tetrads: 85968 edges
  Columns: ['source', 'target', 'max_common']


## Interactive Exploration with Cosmograph

Choose different link types to visualize:

In [ ]:
# Add human-readable label for visualization
nodes_df['label'] = nodes_df.apply(
    lambda r: f"{r['forte_name'] or ''} {r['pcset']}" if r['cardinality'] <= 6 else str(r['pcset']),
    axis=1
)

# Visualization function
def visualize_links(
    links_df: pd.DataFrame,
    title: str = "PC-Set Network",
    filter_cardinality: tuple = None,
    **cosmo_kwargs
):
    """
    Visualize a link dataframe with cosmograph.
    
    Args:
        links_df: Links with 'source' and 'target' columns
        title: Title for the visualization
        filter_cardinality: Optional (min, max) to filter nodes
        **cosmo_kwargs: Additional args passed to cosmo()
    """
    # Get nodes involved in these links
    involved_ids = set(links_df['source']) | set(links_df['target'])
    
    # Filter nodes
    vis_nodes = nodes_df[nodes_df['id_'].isin(involved_ids)].copy()
    
    if filter_cardinality:
        min_c, max_c = filter_cardinality
        vis_nodes = vis_nodes[(vis_nodes['cardinality'] >= min_c) & (vis_nodes['cardinality'] <= max_c)]
        involved_ids = set(vis_nodes['id_'])
        links_df = links_df[links_df['source'].isin(involved_ids) & links_df['target'].isin(involved_ids)]
    
    print(f"{title}: {len(vis_nodes)} nodes, {len(links_df)} edges")
    
    defaults = dict(
        points=vis_nodes,
        links=links_df,
        point_id_by='id_',
        link_source_by='source',
        link_target_by='target',
        point_size_by='cardinality',
        point_color_by='cardinality',
        point_label_by='label',
    )
    defaults.update(cosmo_kwargs)
    
    return cosmo(**defaults)


# Example visualizations:

print("Available visualizations:")
print("  1. visualize_links(immediate_subset_links, 'Subset Lattice')")
print("  2. visualize_links(ti_equiv_links, 'TI-Equivalence Classes')")
print("  3. visualize_links(complement_links, 'Complement Pairs')")
print("  4. visualize_links(z_relation_links, 'Z-Relations')")

Available visualizations:
  1. visualize_links(immediate_subset_links, 'Subset Lattice')
  2. visualize_links(ti_equiv_links, 'TI-Equivalence Classes')
  3. visualize_links(complement_links, 'Complement Pairs')
  4. visualize_links(z_relation_links, 'Z-Relations')


In [17]:
g2 = visualize_links(ti_equiv_links, "TI-Equivalence Classes", filter_cardinality=(3, 6))
g2

TI-Equivalence Classes: 2431 nodes, 24493 edges


Cosmograph(background_color=None, components_display_state_mode=None, focused_point_ring_color=None, hovered_p…

## Save the data to parquet files

In [20]:
print(f"{nodes_df.shape=}")
nodes_df.iloc[0]

nodes_df.shape=(4096, 11)


id_                                 0
pcset                              ()
cardinality                         0
contains_zero                   False
complement_id                    4095
prime_form                         ()
forte_name                       None
is_forte_set                    False
interval_vector    (0, 0, 0, 0, 0, 0)
is_t_symmetric                  False
label                              ()
Name: 0, dtype: object

In [24]:
nodes_df.to_parquet('twelve_tone_sets.parquet')